# SE-801 Lecture 8: Feedback and Learning-System Consolidation

This notebook verifies a scratch MLP against an equivalent `nn.Module`, then
runs one complete train, validation, checkpoint-selection, and test pipeline.

In [ ]:
from __future__ import annotations

import copy
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

SEED = 20261026
ROOT = Path.cwd()
OUTPUT_DIR = ROOT / "outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
print("PyTorch:", torch.__version__)
print("Seed:", SEED)

## 1. Independent synthetic splits and train-only preprocessing

In [ ]:
def make_split(per_class: int, *, seed: int):
    generator = torch.Generator().manual_seed(seed)
    centers = torch.tensor([
        [-1.4, -0.9, 0.5, 0.0],
        [1.2, -0.7, -0.4, 0.8],
        [0.0, 1.3, 0.7, -0.8],
    ])
    scales = torch.tensor([0.55, 0.42, 0.62, 0.48])
    features, targets = [], []
    for class_id, center in enumerate(centers):
        x = center + 1.35 * torch.randn(per_class, 4, generator=generator) * scales
        x[:, 2] += 0.22 * x[:, 0] * x[:, 1]
        features.append(x)
        targets.append(torch.full((per_class,), class_id, dtype=torch.long))
    x, y = torch.cat(features), torch.cat(targets)
    order = torch.randperm(len(y), generator=generator)
    return x[order], y[order]


raw_train_x, train_y = make_split(160, seed=SEED + 1)
raw_val_x, val_y = make_split(55, seed=SEED + 2)
raw_test_x, test_y = make_split(55, seed=SEED + 3)

train_mean = raw_train_x.mean(dim=0)
train_std = raw_train_x.std(dim=0).clamp_min(1e-6)
train_x = (raw_train_x - train_mean) / train_std
val_x = (raw_val_x - train_mean) / train_std
test_x = (raw_test_x - train_mean) / train_std
print("Splits:", train_x.shape, val_x.shape, test_x.shape)
print("Train standardized mean:", train_x.mean(dim=0))

## 2. Scratch tensors and an equivalent module

The scratch matrices use `[input, output]`. `nn.Linear.weight` stores
`[output, input]`, so the copies require transposes.

In [ ]:
dtype = torch.float64
batch_x = train_x[:9].to(dtype)
batch_y = train_y[:9]
generator = torch.Generator().manual_seed(SEED + 10)

W1 = (torch.randn(4, 12, generator=generator, dtype=dtype) * 0.18).requires_grad_()
b1 = torch.zeros(12, dtype=dtype, requires_grad=True)
W2 = (torch.randn(12, 3, generator=generator, dtype=dtype) * 0.18).requires_grad_()
b2 = torch.zeros(3, dtype=dtype, requires_grad=True)


def scratch_forward(x):
    return torch.relu(x @ W1 + b1) @ W2 + b2


class EquivalentMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(4, 12, dtype=dtype)
        self.fc2 = nn.Linear(12, 3, dtype=dtype)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x)))


module = EquivalentMLP()
with torch.no_grad():
    module.fc1.weight.copy_(W1.T)
    module.fc1.bias.copy_(b1)
    module.fc2.weight.copy_(W2.T)
    module.fc2.bias.copy_(b2)

scratch_logits = scratch_forward(batch_x)
module_logits = module(batch_x)
forward_max_error = float((scratch_logits - module_logits).detach().abs().max())

scratch_loss = F.cross_entropy(scratch_logits, batch_y)
module_loss = F.cross_entropy(module_logits, batch_y)
scratch_loss.backward()
module_loss.backward()

gradient_errors = {
    "W1": float((W1.grad - module.fc1.weight.grad.T).abs().max()),
    "b1": float((b1.grad - module.fc1.bias.grad).abs().max()),
    "W2": float((W2.grad - module.fc2.weight.grad.T).abs().max()),
    "b2": float((b2.grad - module.fc2.bias.grad).abs().max()),
}
loss_error = float(abs(scratch_loss.item() - module_loss.item()))
print("Forward max error:", forward_max_error)
print("Loss error:", loss_error)
print("Gradient max errors:", gradient_errors)

## 3. Broadcasting fault demonstration

`[3,1] - [3]` produces a `[3,3]` all-pairs matrix. The code returns a scalar
even though it computes the wrong problem.

In [ ]:
pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([1.0, 2.0, 3.0])
faulty_difference = pred - target
faulty_mse = float(faulty_difference.square().mean())
correct_mse = float((pred.squeeze(1) - target).square().mean())
print("Faulty shape:", tuple(faulty_difference.shape))
print("Faulty MSE:", faulty_mse, "Correct MSE:", correct_mse)
assert faulty_difference.shape == (3, 3)
assert correct_mse == 0.0

## 4. Complete modular training pipeline

In [ ]:
class ConsolidatedMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(4, 12), nn.ReLU(), nn.Linear(12, 3))

    def forward(self, x):
        return self.net(x)


def accuracy(logits, targets):
    return float((logits.argmax(dim=1) == targets).float().mean())


torch.manual_seed(SEED + 20)
model = ConsolidatedMLP()
optimizer = torch.optim.SGD(model.parameters(), lr=0.08, momentum=0.9)
loader_generator = torch.Generator().manual_seed(SEED + 30)
loader = DataLoader(TensorDataset(train_x, train_y), batch_size=48, shuffle=True, generator=loader_generator)

history = []
best = None
best_val_loss = float("inf")
for epoch in range(1, 81):
    model.train()
    for x, y in loader:
        optimizer.zero_grad(set_to_none=True)
        loss = F.cross_entropy(model(x), y)
        loss.backward()
        optimizer.step()
    model.eval()
    with torch.no_grad():
        train_logits = model(train_x)
        val_logits = model(val_x)
        row = {
            "epoch": epoch,
            "train_loss": float(F.cross_entropy(train_logits, train_y)),
            "validation_loss": float(F.cross_entropy(val_logits, val_y)),
            "train_accuracy": accuracy(train_logits, train_y),
            "validation_accuracy": accuracy(val_logits, val_y),
        }
    history.append(row)
    if row["validation_loss"] < best_val_loss:
        best_val_loss = row["validation_loss"]
        best = {"epoch": epoch, "state": copy.deepcopy(model.state_dict())}

model.load_state_dict(best["state"])
model.eval()
with torch.no_grad():
    test_logits = model(test_x)
test_loss = float(F.cross_entropy(test_logits, test_y))
test_accuracy = accuracy(test_logits, test_y)
predicted = test_logits.argmax(dim=1)
confusion = torch.zeros(3, 3, dtype=torch.int64)
for truth, guess in zip(test_y, predicted):
    confusion[truth, guess] += 1
class_recall = confusion.diag().float() / confusion.sum(dim=1).clamp_min(1)
balanced_accuracy = float(class_recall.mean())
parameter_count = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Best epoch:", best["epoch"])
print("Test accuracy:", test_accuracy)
print("Balanced accuracy:", balanced_accuracy)
print("Confusion matrix:\n", confusion)

fig, axes = plt.subplots(1, 2, figsize=(10, 4), constrained_layout=True)
epochs = [row["epoch"] for row in history]
axes[0].plot(epochs, [row["train_loss"] for row in history], label="train", color="#B87521")
axes[0].plot(epochs, [row["validation_loss"] for row in history], label="validation", color="#246B8E")
axes[1].plot(epochs, [row["train_accuracy"] for row in history], label="train", color="#B87521")
axes[1].plot(epochs, [row["validation_accuracy"] for row in history], label="validation", color="#246B8E")
axes[0].set(title="Cross-entropy", xlabel="epoch", ylabel="loss")
axes[1].set(title="Accuracy", xlabel="epoch", ylabel="accuracy", ylim=(0.3, 1.02))
for axis in axes:
    axis.grid(alpha=0.25)
    axis.legend()
fig.savefig(OUTPUT_DIR / "consolidated_learning_curves.png", dpi=170)
plt.close(fig)

## 5. Save the registered evidence

In [ ]:
results = {
    "seed": SEED,
    "protocol": {
        "task": "three-class synthetic four-feature classification",
        "split_sizes": {"train": len(train_y), "validation": len(val_y), "test": len(test_y)},
        "preprocessing": "mean and standard deviation fitted on training features only",
        "architecture": "4-12-3 ReLU MLP",
        "optimizer": "SGD with momentum 0.9",
        "learning_rate": 0.08,
        "batch_size": 48,
        "epochs": 80,
        "selection": "lowest validation cross-entropy",
    },
    "equivalence": {
        "forward_max_error": forward_max_error,
        "loss_error": loss_error,
        "gradient_max_errors": gradient_errors,
    },
    "broadcasting_demo": {
        "faulty_shape": list(faulty_difference.shape),
        "faulty_mse": faulty_mse,
        "correct_mse": correct_mse,
    },
    "training": {
        "history": history,
        "parameter_count": parameter_count,
        "best_epoch": best["epoch"],
        "best_validation_loss": best_val_loss,
        "test_loss": test_loss,
        "test_accuracy": test_accuracy,
        "balanced_accuracy": balanced_accuracy,
        "confusion_matrix": confusion.tolist(),
    },
}
(OUTPUT_DIR / "experiment_results.json").write_text(json.dumps(results, indent=2))
print("Wrote", OUTPUT_DIR / "experiment_results.json")